# PCM — Google Colab Test Notebook

Run the latest **Private Content Manager (PCM)** code from GitHub `main`. This notebook pulls updates before installing dependencies and starting the bot.

**Before you start**
- Add credentials in Colab **Secrets** (key icon in the left sidebar).
- Keep the bot token, database URL, and GoFile API key private.
- Use the bot only from the configured owner's private Telegram chat.
- Colab runtimes are temporary; this is for testing, not 24/7 hosting.

## 1. Add Colab Secrets

In the Colab left sidebar, open **Secrets** and add these exact names, then enable notebook access for each:

**Required**
- `TELEGRAM_BOT_TOKEN`
- `TELEGRAM_API_ID`
- `TELEGRAM_API_HASH`
- `TELEGRAM_OWNER_ID`
- `INSFORGE_DATABASE_URL`

**Optional**
- `GOFILE_TOKEN` — fallback GoFile account token. You can also configure a key interactively in Telegram using `/me` → **GoFile** → **Set API key**. The interactive setting is stored in PCM's database.

Do not paste credentials directly into notebook cells.

In [ ]:
# Pull the latest PCM code from GitHub main. Safe to rerun in an existing Colab runtime.
!if [ -d /content/PCM/.git ]; then cd /content/PCM && git fetch origin main && git pull --ff-only origin main; else cd /content && git clone --branch main https://github.com/Telebotfaroff/PCM.git; fi
%cd /content/PCM
!echo "PCM revision:" && git log -1 --oneline

In [ ]:
# Install project dependencies
!pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y ffmpeg

In [ ]:
# Load credentials from Colab Secrets into environment variables
import os
from google.colab import userdata

REQUIRED_SECRET_NAMES = [
    "TELEGRAM_BOT_TOKEN",
    "TELEGRAM_API_ID",
    "TELEGRAM_API_HASH",
    "TELEGRAM_OWNER_ID",
    "INSFORGE_DATABASE_URL",
]
OPTIONAL_SECRET_NAMES = ["GOFILE_TOKEN"]

missing = []
for name in REQUIRED_SECRET_NAMES + OPTIONAL_SECRET_NAMES:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
        elif name in REQUIRED_SECRET_NAMES:
            missing.append(name)
    except Exception:
        if name in REQUIRED_SECRET_NAMES:
            missing.append(name)

if missing:
    raise RuntimeError(
        "Missing or inaccessible Colab Secrets: "
        + ", ".join(missing)
        + ". Add them in the Secrets panel and enable notebook access."
    )

print("All required secrets loaded. Values are not displayed.")
print("Optional GOFILE_TOKEN:", "configured" if os.environ.get("GOFILE_TOKEN") else "not configured (guest mode or set it in /me)")

## 2. Verify configuration

This checks required environment variables without printing their values. The optional GoFile token can be added in Colab Secrets or through the interactive `/me` menu in Telegram.

In [ ]:
required = [
    "TELEGRAM_BOT_TOKEN",
    "TELEGRAM_API_ID",
    "TELEGRAM_API_HASH",
    "TELEGRAM_OWNER_ID",
    "INSFORGE_DATABASE_URL",
]
for key in required:
    print(f"{key}: {'OK' if os.environ.get(key) else 'MISSING'}")
print(f"GOFILE_TOKEN: {'configured' if os.environ.get('GOFILE_TOKEN') else 'optional / not set'}")

## 3. Start PCM

Run this cell and keep it running while testing Telegram. Stop the cell to stop the bot. The command uses `python -u -m pcm` so startup logs and tracebacks appear immediately.

In [ ]:
# Start the bot in a separate process; startup and incoming-message diagnostics appear below.
%cd /content/PCM
!python -u -m pcm


## 4. Test the interactive Telegram controls

Open the bot's **private chat** using the Telegram account whose numeric ID is `TELEGRAM_OWNER_ID`.

1. Send `/start` or `/help` to check the bot responds.
2. Send `/me` to open the interactive **PCM Control Panel**.
3. Tap **Upload destination** to choose Telegram only, GoFile only, or both.
4. Tap **GoFile** to configure an account API key, switch to guest mode, or clear the saved key. If you set a key through the bot, send it only when prompted; PCM attempts to delete the message after receiving it.
5. Use **Jobs & controls** to inspect recent jobs and open applicable pause, resume, cancel, or retry actions.
6. Use **Auto highest quality**, **Delete download menu**, and **Dump channel upload / Set dump channel** to configure those options.
7. Try `/ytdlp` to see usage, then `/ytdlp YOUR_SUPPORTED_VIDEO_URL` with a URL you are authorized to download.

Not every site or URL is supported. Login-protected or DRM-protected videos may not work. Telegram upload limits differ from GoFile; choose GoFile-only for larger uploads within PCM's configured download limit.

## Troubleshooting

- **Bot does not respond:** Keep the start cell running and inspect the timestamped log output. Send `/start` in the bot's private chat.
- **No incoming-update diagnostics:** Check the bot token, network connection, and that no other process is polling with the same token.
- **Updates appear but no reply:** Check that `TELEGRAM_OWNER_ID` matches your numeric Telegram account ID and use a private chat. PCM intentionally ignores other users.
- **Missing settings / secret error:** Check the required secret names and enable notebook access.
- **GoFile key not working:** Check the key and account permissions. You can use guest mode from `/me` → **GoFile**. Do not post the key in a group chat.
- **No module named `yt_dlp`:** Rerun the dependency-install cell.
- **FFmpeg error:** Rerun the FFmpeg install cell.
- **Unsupported URL:** Try another yt-dlp-supported URL; some sites require authentication or change their protections.

Do not share logs containing credentials or private information. This notebook cannot guarantee that an external site's live download or upload will succeed.